# Northstar Industrial Components Ltd.
## Financial Due Diligence — Expert Demonstration

**Company:** Northstar Industrial Components Ltd.  
**Industry:** Industrial manufacturing and distribution  
**Reporting currency:** USD | **Periods:** Jan 2024 – Dec 2025

**Input files:** `general_ledger.csv` · `monthly_financials.csv` · `accounts_receivable_aging.csv` · `management_adjustments.csv`

---
## Phase 1 — Data Loading and Initial Inspection
**Objective:** Load all four datasets, confirm structure, and establish the financial baseline.  
**Estimated time: 3–4 minutes**

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

# Load all four datasets
# Account_Code is stored as integer in the CSV; read as string for reliable filtering
gl  = pd.read_csv('general_ledger.csv',
                  dtype={'Account_Code': str},
                  parse_dates=['Transaction_Date', 'Posting_Date'])
mf  = pd.read_csv('monthly_financials.csv')
ar  = pd.read_csv('accounts_receivable_aging.csv', parse_dates=['Invoice_Date', 'Due_Date'])
adj = pd.read_csv('management_adjustments.csv')

mf['Period'] = pd.to_datetime(mf['Month'])
mf['Year']   = mf['Period'].dt.year

print('Files loaded successfully.')
print(f'  General Ledger:          {len(gl):,} rows')
print(f'  Monthly Financials:      {len(mf):,} rows')
print(f'  AR Aging:                {len(ar):,} rows')
print(f'  Management Adjustments:  {len(adj):,} rows')

In [ ]:
# Two-year P&L summary from management accounts
summary = mf.groupby('Year').agg(
    Revenue                  = ('Revenue',                  'sum'),
    Gross_Profit             = ('Gross_Profit',             'sum'),
    EBITDA                   = ('EBITDA',                   'sum'),
    Depreciation_Amortization= ('Depreciation_Amortization','sum'),
    Net_Income               = ('Net_Income',               'sum'),
).reset_index()
summary['GP_Margin_%']     = (summary['Gross_Profit'] / summary['Revenue'] * 100).round(1)
summary['EBITDA_Margin_%'] = (summary['EBITDA']       / summary['Revenue'] * 100).round(1)

print('=== Northstar — Summary P&L (Management Accounts) ===')
print(summary.to_string(index=False))
print()
print('Note: EBITDA is reported before Depreciation & Amortization.')
print('D&A is already excluded from EBITDA; any adjustment to D&A does not affect EBITDA directly.')

---
## Phase 2 — Revenue and Transaction Investigation
**Objective:** Reconcile GL revenue against management accounts. Identify the period cutoff issue. Detect any duplicated transactions.  
**Estimated time: 6–7 minutes**

In [ ]:
# Revenue reconciliation: GL posting date vs management accounts
rev_gl = gl[gl['Account_Code'] == '4010'].copy()
rev_gl['Year'] = rev_gl['Posting_Date'].dt.year
gl_by_year = rev_gl.groupby('Year')['Credit'].sum()
mf_by_year = mf.groupby('Year')['Revenue'].sum()

recon = pd.DataFrame({'GL_Revenue': gl_by_year, 'MF_Revenue': mf_by_year})
recon['Variance_GL_minus_MF'] = recon['GL_Revenue'] - recon['MF_Revenue']

print('=== Revenue Reconciliation: GL (by posting date) vs Management Accounts ===')
print(recon.to_string())
print()
print('Interpretation:')
print('  2024: GL revenue is BELOW management accounts — an invoice accrued in Dec 2024')
print('        was posted to the GL in January 2025, leaving GL 2024 understated.')
print('  2025: GL revenue is ABOVE management accounts — that same invoice')
print('        inflates GL 2025 when it should belong to 2024.')

In [ ]:
# FINDING 1: Identify the specific cutoff transaction
# Invoice with 2024 transaction date but 2025 posting date
cutoff_txn = gl[
    (gl['Account_Code'] == '4010') &
    (gl['Transaction_Date'].dt.year == 2024) &
    (gl['Posting_Date'].dt.year == 2025)
]
print('=== FINDING 1 — Revenue Cutoff ===')
print(cutoff_txn[['Transaction_ID', 'Transaction_Date', 'Posting_Date',
                   'Credit', 'Counterparty', 'Document_Reference']].to_string(index=False))
amt = cutoff_txn['Credit'].sum()
print(f'\nAmount: ${amt:,.0f}')
print('Invoice dated 28-Dec-2024 was posted to the GL on 08-Jan-2025.')
print('Management accounts correctly accrue it in December 2024; the GL does not.')
print(f'Impact: 2024 GL revenue understated by ${amt:,.0f}; 2025 GL revenue overstated by the same amount.')
print('This is a timing difference, not a loss. Treatment: reclassify to correct period.')

In [ ]:
# FINDING 5: Duplicate transaction detection
# Same document reference appearing more than once with the same account and debit amount
gl_debits = gl[gl['Debit'] > 0].copy()
dup_counts = (
    gl_debits
    .groupby(['Document_Reference', 'Account_Code', 'Debit'])
    .size().reset_index(name='Posting_Count')
)
flagged = dup_counts[dup_counts['Posting_Count'] > 1]
print('=== Potential Duplicate Postings (same ref + account + amount, posted more than once) ===')
print(flagged.to_string(index=False))

# Drill into the flagged PO
dup_detail = gl[gl['Document_Reference'] == 'PO-2024-0892'].sort_values('Posting_Date')
print('\n=== Detail: PO-2024-0892 ===')
print(dup_detail[['Transaction_ID', 'Transaction_Date', 'Posting_Date',
                   'Account_Code', 'Account_Name', 'Debit', 'Credit',
                   'Counterparty', 'Description']].to_string(index=False))
print()
print('FINDING 5 — Duplicate transaction:')
print('  PO-2024-0892 (TechServ Maintenance, equipment maintenance) was posted TWICE in Oct 2024.')
print('  Each posting debits $42,300 to expenses and credits $42,300 to AP.')
print('  One posting is legitimate; the second is a duplicate.')
print('  Expense overstatement: $42,300 (not $84,600 — the second debit is the error, not both).')
print('  Management proposes ADJ-008 ($42,300) to reverse the duplicate.')
print('  Treatment: accept ADJ-008 if TechServ vendor credit note is confirmed (IR-003).')

---
## Phase 3 — Earnings Quality and Receivables Assessment
**Objective:** Evaluate all 2025 management adjustments individually. Test recurrence of restructuring. Assess the most material overdue receivable.  
**Estimated time: 8–10 minutes**

In [ ]:
# Management EBITDA bridge — 2025 only
ebitda_2025 = mf[mf['Year'] == 2025]['EBITDA'].sum()

# Filter to 2025 adjustments only (Period is stored as integer)
adj_2025 = adj[adj['Period'] == 2025].copy()
total_adj_2025 = adj_2025['Amount'].sum()

print(f'Reported EBITDA 2025:                  ${ebitda_2025:>12,.0f}')
print(f'Management proposed add-backs (2025):  ${total_adj_2025:>12,.0f}')
print(f'Management-adjusted EBITDA 2025:       ${ebitda_2025 + total_adj_2025:>12,.0f}')
print()
print('=== 2025 Adjustment Proposals ===')
print(adj_2025[['Adjustment_ID', 'Description', 'Account_Code',
                'Amount', 'Recurrence_Indicator', 'Supporting_Document_Reference']].to_string(index=False))
print()
print('Note: 2024 adjustments (ADJ-001, ADJ-002, ADJ-008) are NOT included here.')
print('      They affect 2024 quality of earnings, not the 2025 EBITDA baseline.')

In [ ]:
# FINDING 3: Unsupported adjustment — ADJ-005
adj5 = adj[adj['Adjustment_ID'] == 'ADJ-005'].iloc[0]
doc_ref = adj5['Supporting_Document_Reference']   # 'CONS-ERP-2025'
print(f'ADJ-005: "{adj5["Description"]}"')
print(f'Amount:  ${adj5["Amount"]:,.0f}  |  Supporting reference: {doc_ref}')

# Search GL for the claimed document reference
gl_match = gl[gl['Document_Reference'] == doc_ref]
print(f'\nGL entries found for "{doc_ref}": {len(gl_match)}')

# Show what IS in the GL for consulting (different reference)
consult_gl = gl[gl['Account_Code'] == '6030'].nlargest(5, 'Debit')
print('\nLargest SGA/consulting charges actually posted in GL:')
print(consult_gl[['Posting_Date', 'Debit', 'Counterparty',
                   'Document_Reference', 'Description']].to_string(index=False))
print()
print('FINDING 3 — ADJ-005 rejected:')
print(f'  No GL entry exists for reference "{doc_ref}".')
print('  A consulting payment of $55,000 does appear in the GL under CONS-2025-Q4,')
print('  but that is a different amount and a different document reference.')
print('  Management cannot substantiate the $95,000 add-back. Reject ADJ-005.')

In [ ]:
# FINDING 4: Recurring restructuring
# Account 6100 = Restructuring Charges (stored as integer in CSV, cast to str on load)
restr_gl = gl[gl['Account_Code'] == '6100'].copy()
restr_gl['Year'] = restr_gl['Posting_Date'].dt.year

print('=== Restructuring Charges in GL by Year ===')
by_year = restr_gl.groupby('Year')['Debit'].sum().reset_index()
by_year.columns = ['Year', 'Total_Restructuring']
print(by_year.to_string(index=False))

print('\n=== Restructuring GL Entries (detail) ===')
print(restr_gl[['Posting_Date', 'Debit', 'Description',
                 'Document_Reference']].to_string(index=False))

restr_adj = adj[adj['Account_Code'] == 6100]   # integer comparison for adj file
print('\n=== Management Restructuring Adjustments ===')
print(restr_adj[['Adjustment_ID', 'Period', 'Amount',
                  'Recurrence_Indicator', 'Management_Rationale']].to_string(index=False))
print()
print('FINDING 4 — Recurring restructuring:')
print('  Restructuring charges posted in Nov-2024 ($95,000) and Feb-2025 ($78,000).')
print('  Management labels both as "One-time" (ADJ-002 for 2024, ADJ-003 for 2025).')
print('  Two consecutive reporting periods with the same type of charge fail the non-recurring test.')
print('  ADJ-002 (2024): accept — arguably a first occurrence affecting 2024 QoE.')
print('  ADJ-003 (2025): REJECT — a second occurrence within 12 months is recurring by definition.')
print('  EBITDA impact of rejection: -$78,000 vs management claim.')

In [ ]:
# Full 2025 adjustment review — classify each
print('=== 2025 Adjustment Assessment ===')
assessment = [
    ('ADJ-003', 78_000,  'REJECT',   'Recurring restructuring — second occurrence in 12 months'),
    ('ADJ-004', 120_000, 'ACCEPT',   'Owner comp above market — standard normalization'),
    ('ADJ-005', 95_000,  'REJECT',   'Unsupported — no GL evidence for CONS-ERP-2025'),
    ('ADJ-006', 42_000,  'REJECT',   'D&A normalization — EBITDA already excludes D&A; adding back would double-count'),
    ('ADJ-007', 36_000,  'ACCEPT',   'Above-market rent — verifiable normalization'),
    ('ADJ-009', 18_500,  'ACCEPT',   'FX revaluation loss — non-cash, one-time'),
]
rows = []
for adj_id, amt, verdict, rationale in assessment:
    rows.append({'ID': adj_id, 'Amount': amt, 'Verdict': verdict, 'Rationale': rationale})
assess_df = pd.DataFrame(rows)
print(assess_df.to_string(index=False))
print()
accepted_ids = [r[0] for r in assessment if r[2] == 'ACCEPT']
rejected_ids = [r[0] for r in assessment if r[2] == 'REJECT']
adj_2025 = adj[adj['Period'] == 2025]
acc_amt = adj_2025[adj_2025['Adjustment_ID'].isin(accepted_ids)]['Amount'].sum()
rej_amt = adj_2025[adj_2025['Adjustment_ID'].isin(rejected_ids)]['Amount'].sum()
print(f'Accepted adjustments (2025): ${acc_amt:,.0f}')
print(f'Rejected adjustments (2025): ${rej_amt:,.0f}')

In [ ]:
# FINDING 2: Receivables impairment — IronBridge Works
print('=== AR Aging Summary ===')
aging_summary = ar.groupby('Aging_Bucket')['Outstanding_Amount'].agg(
    Count='count', Total='sum'
).reset_index()
print(aging_summary.to_string(index=False))

print('\n=== Invoices > 90 Days Overdue ===')
over90 = ar[ar['Aging_Bucket'] == 'Over 90']
print(over90[['Customer_Name', 'Invoice_Number', 'Outstanding_Amount',
              'Days_Overdue', 'Collection_Notes']].to_string(index=False))

ib = ar[ar['Customer_ID'] == 'C008'].iloc[0]
print()
print('FINDING 2 — IronBridge Works:')
print(f'  Invoice:     {ib["Invoice_Number"]}')
print(f'  Amount:      ${ib["Outstanding_Amount"]:,.0f}')
print(f'  Days overdue: {int(ib["Days_Overdue"])}')
print(f'  Status:      {ib["Collection_Notes"]}')
print('  Recommendation: Full impairment. The customer is in formal administration;')
print('  recovery is highly uncertain. Working capital reduction: $312,800.')

---
## Phase 4 — Findings and Financial Conclusions
**Objective:** Calculate defensible EBITDA. Adjust AR. Consolidate all five findings. Distinguish revenue timing, balance sheet exposures, and EBITDA adjustments.  
**Estimated time: 4–5 minutes**

In [ ]:
# Defensible EBITDA bridge — 2025
ebitda_2025 = mf[mf['Year'] == 2025]['EBITDA'].sum()
adj_2025    = adj[adj['Period'] == 2025]

accepted_ids = ['ADJ-004', 'ADJ-007', 'ADJ-009']
rejected_ids = ['ADJ-003', 'ADJ-005', 'ADJ-006']

acc_amt  = adj_2025[adj_2025['Adjustment_ID'].isin(accepted_ids)]['Amount'].sum()
rej_amt  = adj_2025[adj_2025['Adjustment_ID'].isin(rejected_ids)]['Amount'].sum()
mgmt_adj = adj_2025['Amount'].sum()

print('=== EBITDA Bridge — 2025 ===')
print(f'Reported EBITDA 2025:                        ${ebitda_2025:>12,.0f}')
print(f'+ Management proposed add-backs (2025 only): ${mgmt_adj:>12,.0f}')
print(f'= Management-adjusted EBITDA 2025:           ${ebitda_2025 + mgmt_adj:>12,.0f}')
print()
print(f'  Accepted: ADJ-004 owner comp, ADJ-007 rent, ADJ-009 FX = ${acc_amt:,.0f}')
print(f'  Rejected: ADJ-003 restructuring, ADJ-005 unsupported,')
print(f'            ADJ-006 D&A double-count                       = ${rej_amt:,.0f}')
print()
print(f'= Defensible Normalized EBITDA 2025:         ${ebitda_2025 + acc_amt:>12,.0f}')
print()
print(f'Reduction vs management claim: ${rej_amt:,.0f}')

In [ ]:
# Adjusted AR
ar_reported      = ar['Outstanding_Amount'].sum()
ar_impairment_f2 = 312_800   # IronBridge — full impairment (F2)

ar_adjusted = ar_reported - ar_impairment_f2

print('=== Working Capital — AR Adjustments ===')
print(f'Reported AR:                   ${ar_reported:>12,.0f}')
print(f'Less: IronBridge impairment:   ${ar_impairment_f2:>12,.0f}  (F2 — customer in administration)')
print(f'Adjusted AR:                   ${ar_adjusted:>12,.0f}')
print()
print('Note: The Dec-2024 revenue cutoff (F1) affects the GL vs MF reconciliation')
print('      but does not reduce AR — the invoice is real and expected to be collected.')
print('      It is a period classification issue, not a collectability issue.')

In [ ]:
# Consolidated findings — categorised to avoid double-counting
ebitda_2025 = mf[mf['Year'] == 2025]['EBITDA'].sum()
adj_2025    = adj[adj['Period'] == 2025]
acc_amt     = adj_2025[adj_2025['Adjustment_ID'].isin(['ADJ-004','ADJ-007','ADJ-009'])]['Amount'].sum()

print('=== Consolidated Findings Register ===')
print()
print('REVENUE TIMING (period reclassification — not a loss):')
print(f'  F1  Revenue cutoff — Dec-2024 invoice posted Jan-2025     $148,500')
print(f'      Treatment: reclassify $148,500 from GL 2025 → GL 2024')
print()
print('BALANCE SHEET (working capital / AR):')
print(f'  F2  IronBridge Works — in administration, 407 days overdue $312,800')
print(f'      Treatment: full impairment; reduce AR and record bad debt provision')
print()
print('EBITDA ADJUSTMENTS (affect normalized earnings):')
print(f'  F3  ADJ-005 unsupported — no GL evidence for CONS-ERP-2025  $95,000  REJECTED')
print(f'  F4  ADJ-003 recurring restructuring                           $78,000  REJECTED')
print(f'      Total rejected 2025 add-backs:                           $173,000')
print()
print('INCOME STATEMENT (expense accuracy):')
print(f'  F5  Duplicate maintenance charge — PO-2024-0892              $42,300')
print(f'      Treatment: accept ADJ-008 if TechServ credit note confirmed (open item)')
print()
print('SUMMARY (no double-counting):')
print(f'  Defensible EBITDA 2025:         ${ebitda_2025 + acc_amt:>12,.0f}')
print(f'  Adjusted AR:                    ${ar['Outstanding_Amount'].sum() - 312_800:>12,.0f}')
print(f'  EBITDA reduction vs mgmt:              $173,000  (rejected adjustments)')
print(f'  Confirmed BS exposure:                 $312,800  (IronBridge impairment)')
print(f'  Conditional EBITDA benefit (F5):        $42,300  (pending credit note)')

In [ ]:
# Executive summary
ebitda_2024 = mf[mf['Year'] == 2024]['EBITDA'].sum()
ebitda_2025 = mf[mf['Year'] == 2025]['EBITDA'].sum()
rev_2024    = mf[mf['Year'] == 2024]['Revenue'].sum()
rev_2025    = mf[mf['Year'] == 2025]['Revenue'].sum()
adj_2025    = adj[adj['Period'] == 2025]
acc_amt     = adj_2025[adj_2025['Adjustment_ID'].isin(['ADJ-004','ADJ-007','ADJ-009'])]['Amount'].sum()
mgmt_adj    = adj_2025['Amount'].sum()
ar_reported = ar['Outstanding_Amount'].sum()

print('=' * 62)
print('NORTHSTAR INDUSTRIAL COMPONENTS LTD.')
print('FINANCIAL DUE DILIGENCE — EXECUTIVE SUMMARY')
print('=' * 62)
print(f'Revenue 2024 (management):            ${rev_2024:>12,.0f}')
print(f'Revenue 2025 (management):            ${rev_2025:>12,.0f}')
print()
print(f'EBITDA 2024 (reported):               ${ebitda_2024:>12,.0f}')
print(f'EBITDA 2025 (reported):               ${ebitda_2025:>12,.0f}')
print(f'EBITDA 2025 (management-adjusted):    ${ebitda_2025+mgmt_adj:>12,.0f}')
rej_amt = adj_2025[adj_2025['Adjustment_ID'].isin(['ADJ-003','ADJ-005','ADJ-006'])]['Amount'].sum()
print(f'EBITDA 2025 (defensible):             ${ebitda_2025+acc_amt:>12,.0f}')
print(f'  Reduction vs mgmt (total rejected):       ${rej_amt:,.0f}')
print(f'  of which F3+F4 findings:                  $173,000  (ADJ-003+ADJ-005)')
print()
print(f'Reported AR:                          ${ar_reported:>12,.0f}')
print(f'Adjusted AR (less IronBridge):        ${ar_reported - 312_800:>12,.0f}')
print()
print('KEY OPEN ITEMS:')
print('  IR-001  IronBridge: confirm administration status and recovery prospects')
print('  IR-002  ADJ-005: provide invoices supporting CONS-ERP-2025')
print('  IR-003  F5 duplicate: obtain TechServ credit note for PO-2024-0892')
print('  IR-004  Restructuring: provide 2026 plan to confirm non-recurrence')
print('=' * 62)